# L2 step 4 · The winning 3B on the benchmark's six tasks (GPU T4)

Task **L2** (roadmap), objective **O4**. Step 3 chose the winner by the rule fixed on 8 Nov; only the winner is scored here, once.

**What it does:** the winning 3B (base `Qwen/Qwen2.5-VL-3B-Instruct` with its adapter merged in) answers the benchmark's 10,675 optical questions of the six tasks with L0 step 2's prompts and scorer, greedy, images at the 3B's training size. Answers are written as they come, so a stopped run continues.

**Reported beside accuracy**
- The new 7B at full size (L0 step 2) and the base 7B (the DisasterM3 paper), for scale.
- Building counting: count error against the true count, and the rule that never sees the image (66.98% on these questions), which accuracy on these options can't rule out.
- E5's clean set (its 186 copied questions dropped) is scored on the PC from `per_question.csv`, as L0 step 2's was.

**Expectation, fixed before the run (8 Nov):** below the new 7B on most tasks, since the 3B is smaller; no number was predicted.

**How to run on Kaggle** (account 2)
1. Settings: GPU **T4 x2** (one is used), internet ON. Secret `HF_TOKEN` (AbrarAlam: the adapters are in a private repo).
2. Inputs: `redwannewaz/disasterm3-bench-mirror-v2`; Your Work → `l2-step3-heldout-select` (its `selection.json` names the winner).
3. Run All once with `SMOKE_TEST = True` (5 questions per task). Then `False` and Save & Run All. About 2–3 h (estimated). It stops cleanly at 11 h; to continue, attach that version's output and run again.
4. Outputs in `/kaggle/working/L2S4/`: `results.csv`, `per_question.csv`, `README.md`, `config.json`, `answers.jsonl`.


In [ ]:
# ── Cell 1: Environment: Kaggle's torch and transformers, current peft (as L2 step 2), and no torchao ──
# L0's merge (26 Oct) stopped on "Found an incompatible version of torchao": peft checks torchao when it puts an adapter
# on an fp16 model, as this notebook does. Nothing here uses torchao.
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 26 Oct)"
import torch
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 1, "Pick GPU T4 x2 (one is used)"


In [ ]:
# ── Cell 2: Configuration: the winner, the benchmark ─────────────────────────
import ast, glob, json, math, os, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoProcessor

BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_REPO = "AbrarAlam/disasterm3-qwen25vl3b-l2"
BATCH, MAX_NEW_TOKENS = 8, 32
SMOKE_TEST = False             # True = 5 questions per task
TIME_LIMIT_HOURS = 11.0
OUT_DIR = Path("/kaggle/working/L2S4")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS_PATH = OUT_DIR / "answers.jsonl"
CUDA = torch.cuda.is_available()
EXPECTED_QUESTIONS = {"DTR": 353, "BBR": 2006, "BDC": 3583, "DRE": 1868, "DSR": 2007, "ORR": 858}
REF_NEW_7B_FULL = {"DTR": 76.77, "BBR": 19.99, "BDC": 30.31, "DRE": 33.03, "DSR": 40.46, "ORR": 34.62}   # L0 step 2
REF_BASE_7B = {"DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "DSR": 28.3, "ORR": 23.9}              # DisasterM3 Table 2

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})
# ── The benchmark's prompts and scorer, and the 3Bs (the same code in steps 3 and 4) ──
# The four prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)
# (code, subset name in the 499A scorer, task name in the benchmark, task name in train_release.json)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", "Disaster Type Recognition"),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", "Disaster Bearing Bodies Recognition"),
         ("BDC", "building_damage_counting", "Building Damage Counting", "Building Damage Counting"),
         ("DRE", "road_damage_counting", "Road Damage Counting", "Road Damage Counting"),
         ("DSR", "landuse", "Disaster Scene Recognition", "Disaster Scene Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", "relational reasoning")]
COUNTING = ("BDC", "DRE")

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def prompt_text(e, subset):
    """(the benchmark prompt, the image keys in order), as L0 step 2's build() makes them."""
    p = e.get("prompts", "")
    p = p[0] if isinstance(p, list) else p
    if subset == "landuse":
        return PROMPTS["landuse"].format(prompt=p, options_str=e["options_str"]), ["pre_image_path"]
    if subset == "relational_reasoning_qa":           # its entries spell the options' field option_str
        return PROMPTS[subset].format(prompt=p, options_str=e["option_str"]), ["image_path"]
    key = "bearing_body" if subset == "bearing_body" else "single_choice"
    return PROMPTS[key].format(prompt=p, options_str=e["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(text, n_images):
    """L0 step 2's layout: single-image prompts put the image first; two-image prompts fill each '<image>' gap."""
    parts = text.split("<image>")
    if len(parts) == 1:
        return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": text}]
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content

def option_values(e):
    """The five option values of a counting question, as numbers ('5.22%' -> 5.22)."""
    o = e["options_list"]
    o = ast.literal_eval(o) if isinstance(o, str) else o
    return [float(str(x).rstrip("%")) for x in o]

def no_image_rule(values):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest (benchmark check, 7 Nov)."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

def chosen_value(answer, values):
    """The value of the one option an answer names; None if it names no option or several."""
    got = letters(answer)
    if len(got) != 1 or next(iter(got)) not in "ABCDE"[:len(values)]:
        return None
    return values["ABCDE".index(next(iter(got)))]

assert letters("The answer is D.") == {"D"} and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert chosen_value("B", [1, 2, 3, 4, 5]) == 2 and chosen_value("B, C", [1, 2, 3, 4, 5]) is None
assert content_of("x<image>y<image>z", 2)[1] == {"type": "image"} and content_of("q", 1)[0] == {"type": "image"}

def load_3b(which):
    """The base 3B in fp16 on one T4; for 'b' or 'c', its finished adapter merged in."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": 0} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return model.eval()

@torch.inference_mode()
def answer_batch(model, items):
    """items: (prompt text, image paths) -> greedy answers, every sampling setting given, images at the 3B's cap."""
    texts, images = [], []
    for text, paths in items:
        imgs = [sized(Image.open(p).convert("RGB")) for p in paths]
        texts.append(processor.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                                   add_generation_prompt=True))
        images += imgs
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

def answer_safe(model, items):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, items)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""]
        half = len(items) // 2
        return answer_safe(model, items[:half]) + answer_safe(model, items[half:])

selection = json.loads(one("L2S3/selection.json", "attach Your Work -> l2-step3-heldout-select").read_text(encoding="utf-8"))
WINNER = selection["winner"]
assert WINNER in ("b", "c"), "step 3's output holds no winner: was it a smoke test?"
BENCH_JSON = one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2")
processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.tokenizer.padding_side = "left"
print(f"✓ L2 step 4{'  [SMOKE TEST]' if SMOKE_TEST else ''}: the winner is ({WINNER}): {selection['reason']}")


In [ ]:
# ── Cell 3: The benchmark's optical questions of the six tasks (as L0 step 2 asked them) ──
with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bench_index = {}                # every benchmark image by file name (L0 step 2's rule: mirrors nest them differently)
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

def sensor(e):
    """H1 V2's rule for the after-image's sensor: most entries have post_image_type, ORR entries image_type."""
    return e.get("post_image_type") or e.get("image_type") or "Optical"

rows = []
for code, subset, bench_task, _ in TASKS:      # ids as in H1 V2, L0 step 2 and E5: the index among the task's entries
    entries = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == bench_task)]
    entries = [(qid, e) for qid, e in entries if sensor(e) == "Optical"]
    assert len(entries) == EXPECTED_QUESTIONS[code], f"{code}: {len(entries)} optical questions"
    for qid, e in entries:
        text, keys = prompt_text(e, subset)
        row = {"id": qid, "task": code, "text": text, "images": [bench_index.get(name_of(e[k])) for k in keys], "truth": e["ground_truth_option"]}
        if code in COUNTING:
            values = option_values(e)
            right = sorted(letters(e["ground_truth_option"]))[0]
            row.update(values=values, truth_value=values["ABCDE".index(right)], rule_correct=no_image_rule(values) == right)
        rows.append(row)
Q = pd.DataFrame(rows)
Q["image_missing"] = Q["images"].map(lambda ps: any(p is None for p in ps))
assert Q["image_missing"].sum() <= 0.01 * len(Q), f"{Q['image_missing'].sum()} questions have an image missing: attach the bench mirror"
print(f"✓ {len(Q):,} optical questions { {k: int(v) for k, v in Q['task'].value_counts().items()} }; "
      f"{int(Q['image_missing'].sum())} with an image missing "
      "(counted wrong)")
if SMOKE_TEST:
    Q = Q.groupby("task").head(5).reset_index(drop=True)


In [ ]:
# ── Cell 4: The winner answers, written as it goes (resumable) ──────────────────
from tqdm.auto import tqdm

previous = sorted(glob.glob("/kaggle/input/**/L2S4/answers.jsonl", recursive=True))
if previous and not ANSWERS_PATH.exists():
    shutil.copy(previous[0], ANSWERS_PATH)
done = {}
if ANSWERS_PATH.exists():
    with open(ANSWERS_PATH, encoding="utf-8") as f:
        done = {r["id"]: r["answer"] for r in map(json.loads, f) if r.get("model") == WINNER}
todo = Q[~Q["id"].isin(list(done)) & ~Q["image_missing"]]
print(f"{len(done):,} already answered; {len(todo):,} to do")
model = load_3b(WINNER)
deadline, t0, stopped = time.time() + TIME_LIMIT_HOURS * 3600, time.time(), False
with open(ANSWERS_PATH, "a", encoding="utf-8") as fh:
    for start in tqdm(range(0, len(todo), BATCH), desc=f"3B ({WINNER})"):
        if time.time() > deadline:
            stopped = True
            break
        batch = todo.iloc[start:start + BATCH]
        for qid, answer in zip(batch["id"], answer_safe(model, list(zip(batch["text"], batch["images"])))):
            fh.write(json.dumps({"id": qid, "model": WINNER, "answer": answer}) + "\n")
            done[qid] = answer
        fh.flush()
RUN_HOURS = (time.time() - t0) / 3600
print(f"{'Time limit reached: stopped cleanly' if stopped else 'All answered'} after {RUN_HOURS:.2f} h")


In [ ]:
# ── Cell 5: Scores against the references, count error, and the output files ──
Q["answer"] = Q["id"].map(done)
Q["correct"] = [a is not None and letters(a) == letters(t) for a, t in zip(Q["answer"], Q["truth"])]
complete = bool((Q["answer"].notna() | Q["image_missing"]).all())
table = []
for code, _, _, _ in TASKS:
    g = Q[Q["task"] == code]
    table.append({"task": code, "questions": len(g), "answered": int(g["answer"].notna().sum()),
                  f"3B_{WINNER}_%": round(100 * g["correct"].mean(), 2) if len(g) else float("nan"),
                  "new_7B_full_%": REF_NEW_7B_FULL[code], "base_7B_paper_%": REF_BASE_7B[code]})
results = pd.DataFrame(table)
bdc = Q[Q["task"] == "BDC"].copy()
bdc["value"] = [chosen_value(a, v) if a is not None else None for a, v in zip(bdc["answer"], bdc["values"])]
err = (bdc["value"].astype(float) - bdc["truth_value"]).abs() / bdc["truth_value"].clip(lower=1)
counting = {"accuracy_%": round(100 * float(bdc["correct"].mean()), 2),
            "median_count_error_%": round(100 * float(err.median()), 1) if err.notna().any() else None,
            "answered_with_one_option_%": round(100 * float(bdc["value"].notna().mean()), 1),
            "no_image_rule_%": round(100 * float(bdc["rule_correct"].mean()), 2)}
print(results.to_string(index=False))
print(f"\nBuilding counting: {counting}")

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

Q.drop(columns=["text"]).to_csv(OUT_DIR / "per_question.csv", index=False)
results.to_csv(OUT_DIR / "results.csv", index=False)
config = {"task": "L2 step 4: the winning 3B on the benchmark", "smoke_test": SMOKE_TEST, "winner": WINNER, "selection": selection,
          "base_model": BASE_MODEL, "adapter": f"{ADAPTER_REPO}/{WINNER}/final", "max_pixels": MAX_PIXELS, "decoding": "greedy",
          "complete": complete, "hours_this_session": round(RUN_HOURS, 2), "building_counting": counting, "versions": VERSIONS}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    f"# L2 step 4: the winning 3B, option ({WINNER}), on the benchmark" + (" (SMOKE TEST)" if SMOKE_TEST else ""), "",
    f"`{BASE_MODEL}` with the adapter `{ADAPTER_REPO}/{WINNER}/final` merged in, on the benchmark's optical questions of the six tasks, "
    f"with L0 step 2's prompts and scorer, greedy, images at most {MAX_PIXELS:,} pixels. Step 3's reason for the winner: {selection['reason']}.", "",
    "## Accuracy (%)", "", as_markdown(results), "",
    "The new 7B's numbers are at full size (1036 px); the 3B reads its training size. The base 7B's are the DisasterM3 paper's (Table 2).", "",
    "## Building counting, beside accuracy", "",
    f"- Accuracy {counting['accuracy_%']}%.",
    f"- Median count error {counting['median_count_error_%']}% (the chosen option's value against the true count), on the "
    f"{counting['answered_with_one_option_%']}% answered with one option.",
    f"- The rule that never sees the image: {counting['no_image_rule_%']}%. Accuracy on these options can't show that a model counts well.", "",
    "## Files", "", "- `results.csv`, `per_question.csv`, `answers.jsonl` (every answer), `config.json`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT_DIR}: results.csv, per_question.csv, answers.jsonl, README.md, config.json"
      + ("" if complete else " (not complete: attach this output and run again)"))
